In [0]:
from pyspark.sql import functions as F

silver_df = spark.table(
    "workspace.video_game_sales.silver_video_game_sales"
)

key_columns = ["name", "platform", "year_raw", "publisher"]

duplicate_keys = (
    silver_df
    .groupBy(*key_columns)
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate business keys:", duplicate_keys.count())

display(duplicate_keys.limit(20))

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table(
    "workspace.video_game_sales.silver_video_game_sales"
)

duplicate_keys = (
    silver_df
    .groupBy("name", "platform", "year_raw", "publisher")
    .count()
    .filter(F.col("count") > 1)
)

display(
    silver_df.join(
        duplicate_keys.select(
            "name", "platform", "year_raw", "publisher"
        ),
        ["name", "platform", "year_raw", "publisher"],
        "inner"
    ).orderBy("name", "platform")
)

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table(
    "workspace.video_game_sales.silver_video_game_sales"
)

# Check null ranks
null_ranks = silver_df.filter(
    F.col("rank").isNull()
).count()

# Check duplicate ranks
duplicate_ranks = (
    silver_df
    .groupBy("rank")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_rank_count = duplicate_ranks.count()

print("Total records:", silver_df.count())
print("Null ranks:", null_ranks)
print("Duplicate rank values:", duplicate_rank_count)

assert null_ranks == 0, "Null ranks detected"
assert duplicate_rank_count == 0, "Duplicate ranks detected"

print("SOURCE RECORD KEY VALIDATION PASSED")

In [0]:

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from uuid import uuid4
from datetime import datetime, timezone
from src.utilities.fingerprints import add_record_fingerprint

# Source and target tables
source_table = "workspace.video_game_sales.silver_video_game_sales"

target_table = "workspace.video_game_sales.incremental_video_game_sales"

audit_table = "workspace.video_game_sales.incremental_processing_audit"

# Read source data
source_df = spark.table(source_table)

# Business columns used to detect changes
hash_columns = [
    "rank",
    "name",
    "platform",
    "year_raw",
    "genre",
    "publisher",
    "na_sales",
    "eu_sales",
    "jp_sales",
    "other_sales",
    "global_sales",
    "year",
    "year_status",
    "regional_sales_total",
    "sales_difference",
    "sales_reconciliation_status"
]

# Generate deterministic record fingerprint
source_df = add_record_fingerprint(
    source_df,
    hash_columns
)

source_count = source_df.count()

# Create an empty Delta target if it does not exist
if not spark.catalog.tableExists(target_table):

    (
        source_df.limit(0)
        .write.format("delta")
        .saveAsTable(target_table)
    )

# Create audit table if it does not exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {audit_table} (
    run_id STRING,
    batch_id STRING,
    source_count BIGINT,
    inserted_count BIGINT,
    updated_count BIGINT,
    unchanged_count BIGINT,
    execution_timestamp TIMESTAMP,
    status STRING
)
USING DELTA
""")

print("Incremental target prepared:", target_table)
print("Source records:", source_count)
print("Target records:", spark.table(target_table).count())
print("Preparation completed.")


In [0]:

# Generate batch identifiers
run_id = str(uuid4())
batch_id = f"silver_snapshot_{run_id}"

# Read current target
target_df = spark.table(target_table)

# Identify new records
new_records = (
    source_df.alias("s")
    .join(
        target_df.select("rank").alias("t"),
        F.col("s.rank") == F.col("t.rank"),
        "left_anti"
    )
)

inserted_count = new_records.count()

# Identify changed records
changed_records = (
    source_df.alias("s")
    .join(
        target_df.alias("t"),
        F.col("s.rank") == F.col("t.rank"),
        "inner"
    )
    .filter(
        F.col("s.record_hash") != F.col("t.record_hash")
    )
)

updated_count = changed_records.count()

unchanged_count = (
    source_count - inserted_count - updated_count
)

# Execute Delta MERGE
delta_target = DeltaTable.forName(spark, target_table)

update_values = {
    column: f"s.`{column}`"
    for column in source_df.columns
}

insert_values = {
    column: f"s.`{column}`"
    for column in source_df.columns
}

(
    delta_target.alias("t")
    .merge(
        source_df.alias("s"),
        "t.rank = s.rank"
    )
    .whenMatchedUpdate(
        condition="t.record_hash <> s.record_hash",
        set=update_values
    )
    .whenNotMatchedInsert(
        values=insert_values
    )
    .execute()
)

# Record execution audit
audit_record = [(
    run_id,
    batch_id,
    source_count,
    inserted_count,
    updated_count,
    unchanged_count,
    datetime.now(timezone.utc),
    "SUCCESS"
)]

audit_schema = """
run_id STRING,
batch_id STRING,
source_count LONG,
inserted_count LONG,
updated_count LONG,
unchanged_count LONG,
execution_timestamp TIMESTAMP,
status STRING
"""

audit_df = spark.createDataFrame(
    audit_record,
    schema=audit_schema
)

audit_df.write.format("delta").mode("append").saveAsTable(
    audit_table
)

print("INCREMENTAL MERGE COMPLETED")
print("Source records:", source_count)
print("Inserted:", inserted_count)
print("Updated:", updated_count)
print("Unchanged:", unchanged_count)


In [0]:

target_df = spark.table(target_table)

target_count = target_df.count()
distinct_ranks = target_df.select("rank").distinct().count()

print("TARGET RECORDS:", target_count)
print("DISTINCT RANKS:", distinct_ranks)

assert target_count == source_count
assert distinct_ranks == target_count

print("\nINCREMENTAL PROCESSING VALIDATION PASSED")

display(
    spark.table(audit_table)
    .orderBy(F.desc("execution_timestamp"))
)


In [0]:

from pyspark.sql import functions as F

# Read the current Silver source
test_source = spark.table(
    "workspace.video_game_sales.silver_video_game_sales"
)

# Select one existing record to simulate an update
update_record = (
    test_source
    .filter(F.col("rank") == 1)
    .withColumn("global_sales", F.lit(83.00))
)

# Select one new record using an unused rank
new_record = (
    test_source
    .filter(F.col("rank") == 1)
    .withColumn("rank", F.lit(999999))
    .withColumn("name", F.lit("Incremental Test Game"))
    .withColumn("global_sales", F.lit(15.50))
)

# Keep all remaining original records unchanged
unchanged_records = test_source.filter(
    F.col("rank") != 1
)

# Combine test batch
test_batch = (
    unchanged_records
    .unionByName(update_record)
    .unionByName(new_record)
)

# Recalculate record fingerprint using reusable utility
test_batch = add_record_fingerprint(
    test_batch,
    hash_columns
)

test_source_table = (
    "workspace.video_game_sales.incremental_test_source"
)

(
    test_batch.write.format("delta")
    .mode("overwrite")
    .saveAsTable(test_source_table)
)

print("Test batch records:", test_batch.count())
print("Test source table:", test_source_table)
print("Original Silver table remains unchanged.")


In [0]:

# Read controlled test batch
test_source_df = spark.table(
    "workspace.video_game_sales.incremental_test_source"
)

test_source_count = test_source_df.count()

# Compare with current incremental target
current_target = spark.table(target_table)

new_records = (
    test_source_df.alias("s")
    .join(
        current_target.select("rank").alias("t"),
        F.col("s.rank") == F.col("t.rank"),
        "left_anti"
    )
)

test_inserted = new_records.count()

changed_records = (
    test_source_df.alias("s")
    .join(
        current_target.alias("t"),
        F.col("s.rank") == F.col("t.rank"),
        "inner"
    )
    .filter(
        F.col("s.record_hash") != F.col("t.record_hash")
    )
)

test_updated = changed_records.count()

test_unchanged = (
    test_source_count - test_inserted - test_updated
)

# Execute Delta MERGE
delta_target = DeltaTable.forName(spark, target_table)

update_values = {
    column: f"s.`{column}`"
    for column in test_source_df.columns
}

insert_values = {
    column: f"s.`{column}`"
    for column in test_source_df.columns
}

(
    delta_target.alias("t")
    .merge(
        test_source_df.alias("s"),
        "t.rank = s.rank"
    )
    .whenMatchedUpdate(
        condition="t.record_hash <> s.record_hash",
        set=update_values
    )
    .whenNotMatchedInsert(
        values=insert_values
    )
    .execute()
)

print("CONTROLLED MERGE COMPLETED")
print("Source records:", test_source_count)
print("Inserted:", test_inserted)
print("Updated:", test_updated)
print("Unchanged:", test_unchanged)


In [0]:

# Verify updated record
display(
    spark.table(target_table)
    .filter(F.col("rank") == 1)
    .select("rank", "name", "global_sales", "record_hash")
)

# Verify inserted record
display(
    spark.table(target_table)
    .filter(F.col("rank") == 999999)
    .select("rank", "name", "global_sales", "record_hash")
)

# Verify original Silver remains unchanged
print(
    "Original Silver records:",
    spark.table(source_table).count()
)

print(
    "Incremental target records:",
    spark.table(target_table).count()
)


In [0]:
print(
    "Original Silver records:",
    spark.table(source_table).count()
)

print(
    "Incremental target records:",
    spark.table(target_table).count()
)

print(
    "Original Wii Sports sales:",
    spark.table(source_table)
    .filter(F.col("rank") == 1)
    .select("global_sales")
    .first()[0]
)

In [0]:

from pyspark.sql import functions as F
from datetime import datetime, timezone
from uuid import uuid4

# ------------------------------------------
# 1. RECONCILIATION
# ------------------------------------------

target_df = spark.table(target_table)
silver_df = spark.table(source_table)

target_count = target_df.count()
distinct_rank_count = target_df.select("rank").distinct().count()
silver_count = silver_df.count()

assert target_count == 16599
assert distinct_rank_count == target_count
assert silver_count == 16598

# Confirm controlled changes
updated_sales = (
    target_df.filter(F.col("rank") == 1)
    .select("global_sales").first()[0]
)

inserted_game = (
    target_df.filter(F.col("rank") == 999999).count()
)

assert updated_sales == 83.00
assert inserted_game == 1

print("RECONCILIATION PASSED")
print("Target records:", target_count)
print("Distinct ranks:", distinct_rank_count)
print("Original Silver records:", silver_count)


# ------------------------------------------
# 2. AUDIT CONTROLLED MERGE
# ------------------------------------------

audit_run_id = str(uuid4())

audit_record = [(
    audit_run_id,
    "controlled_test_batch",
    test_source_count,
    test_inserted,
    test_updated,
    test_unchanged,
    datetime.now(timezone.utc),
    "SUCCESS"
)]

audit_schema = """
run_id STRING,
batch_id STRING,
source_count LONG,
inserted_count LONG,
updated_count LONG,
unchanged_count LONG,
execution_timestamp TIMESTAMP,
status STRING
"""

controlled_audit_df = spark.createDataFrame(
    audit_record,
    schema=audit_schema
)

controlled_audit_df.write.format("delta") \
    .mode("append") \
    .saveAsTable(audit_table)

print("CONTROLLED MERGE AUDIT LOGGED")

display(
    spark.table(audit_table)
    .orderBy(F.desc("execution_timestamp"))
)


In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

silver_df = spark.table(
    "workspace.video_game_sales.silver_video_game_sales"
)

restored_df = add_record_fingerprint(
    silver_df,
    hash_columns
)

# Replace incremental target with clean baseline
(
    restored_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

# Remove temporary test source
spark.sql(f"DROP TABLE IF EXISTS {test_source_table}")

# Verify restoration
final_target = spark.table(target_table)

final_count = final_target.count()
final_distinct_ranks = final_target.select("rank").distinct().count()

original_sales = (
    final_target
    .filter(F.col("rank") == 1)
    .select("global_sales")
    .first()[0]
)

assert final_count == 16598
assert final_distinct_ranks == 16598
assert original_sales == 82.74

print("CLEANUP COMPLETED")
print("Final target records:", final_count)
print("Distinct ranks:", final_distinct_ranks)
print("Original Wii Sports sales:", original_sales)
print("Temporary test source removed.")

print("\nINCREMENTAL PIPELINE BASELINE RESTORED")
